In [40]:
#imports 
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
from sklearn.preprocessing import OrdinalEncoder
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.model_selection import GridSearchCV

In [11]:
df=pd.read_csv("road_network.csv")
print(df.index)
print("All columns are")
ai=0
for i in df:
    print(ai,i)
    ai+=1

RangeIndex(start=0, stop=200, step=1)
All columns are
0 road_id
1 road_name
2 road_type
3 length_km
4 lanes
5 speed_limit_kmh
6 capacity_vehicles_per_hour
7 has_bus_lane
8 has_bike_lane
9 surface_condition
10 last_maintenance_date
11 latitude_start
12 longitude_start
13 latitude_end
14 longitude_end
15 district
16 toll_rate_rp
17 avg_daily_traffic


In [12]:
df.shape

(200, 18)

In [13]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 200 entries, 0 to 199
Data columns (total 18 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   road_id                     200 non-null    object 
 1   road_name                   200 non-null    object 
 2   road_type                   200 non-null    object 
 3   length_km                   200 non-null    float64
 4   lanes                       200 non-null    int64  
 5   speed_limit_kmh             200 non-null    int64  
 6   capacity_vehicles_per_hour  200 non-null    int64  
 7   has_bus_lane                200 non-null    int64  
 8   has_bike_lane               200 non-null    int64  
 9   surface_condition           200 non-null    object 
 10  last_maintenance_date       200 non-null    object 
 11  latitude_start              200 non-null    float64
 12  longitude_start             200 non-null    float64
 13  latitude_end                200 non

In [14]:
for i in df.select_dtypes(include=['object', 'string']):
    counts = df[i].value_counts()
    print(counts)

road_id
RD-0001    1
RD-0138    1
RD-0128    1
RD-0129    1
RD-0130    1
          ..
RD-0070    1
RD-0071    1
RD-0072    1
RD-0073    1
RD-0200    1
Name: count, Length: 200, dtype: int64
road_name
Jl. Segment 1      1
Jl. Segment 138    1
Jl. Segment 128    1
Jl. Segment 129    1
Jl. Segment 130    1
                  ..
Jl. Segment 70     1
Jl. Segment 71     1
Jl. Segment 72     1
Jl. Segment 73     1
Jl. Segment 200    1
Name: count, Length: 200, dtype: int64
road_type
Jalan Tol           48
Jalan Lingkungan    46
Jalan Kolektor      44
Jalan Lokal         33
Jalan Arteri        29
Name: count, dtype: int64
surface_condition
Baik            106
Cukup            61
Rusak Ringan     23
Rusak Berat      10
Name: count, dtype: int64
last_maintenance_date
2022-08-10    3
2023-02-05    3
2022-12-21    2
2022-12-18    2
2023-12-23    2
             ..
2023-05-01    1
2022-09-16    1
2022-06-03    1
2023-07-30    1
2024-01-01    1
Name: count, Length: 173, dtype: int64
district
Mangga Du

In [35]:
# Features and target seperation
X = df.drop(
    columns=[
        'avg_daily_traffic',
        'last_maintenance_date',
        'road_id',
        'road_name'
    ],
    errors='ignore'
)

y = df['avg_daily_traffic']


In [36]:
# Training a general random forest
categorical_cols = X.select_dtypes(include='object').columns


preprocessor = ColumnTransformer([
    ('cat',
     OneHotEncoder(drop='first', handle_unknown='ignore'),
     categorical_cols)
], remainder='passthrough')


rf_pipe = Pipeline([
    ('preprocess', preprocessor),
    ('model', RandomForestRegressor(
        n_estimators=200,
        random_state=42,
        n_jobs=-1
    ))
])


In [37]:

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)


rf_pipe.fit(X_train, y_train)


print("Train R²:", rf_pipe.score(X_train, y_train))
print("Test R²:", rf_pipe.score(X_test, y_test))


Train R²: 0.9776422717895181
Test R²: 0.839459557349774


In [38]:
# Cross Validation

cv_scores = cross_val_score(
    rf_pipe,
    X,
    y,
    cv=5,
    scoring='r2',
    n_jobs=-1
)

print("\nCV Scores:", cv_scores)
print("Mean CV R²:", cv_scores.mean())


CV Scores: [0.78869393 0.78379203 0.81961197 0.85236833 0.82407679]
Mean CV R²: 0.8137086118124731


In [41]:
# Hyperparameter Tuning

params = {
    'model__n_estimators': [100, 200, 300],
    'model__max_depth': [None, 5, 10, 15],
    'model__min_samples_split': [2, 5, 10],
    'model__min_samples_leaf': [1, 2, 4]
}

grid = GridSearchCV(
    rf_pipe,
    params,
    cv=5,
    scoring='r2',
    n_jobs=-1
)

grid.fit(X_train, y_train)

print("Best Params:", grid.best_params_)
print("Best CV Score:", grid.best_score_)

Best Params: {'model__max_depth': 5, 'model__min_samples_leaf': 1, 'model__min_samples_split': 5, 'model__n_estimators': 100}
Best CV Score: 0.8342341127172095


In [44]:
results = pd.DataFrame(grid.cv_results_)

top_results = (
    results[
        [
            'mean_test_score',
            'param_model__n_estimators',
            'param_model__max_depth',
            'param_model__min_samples_split',
            'param_model__min_samples_leaf'
        ]
    ]
    .sort_values('mean_test_score', ascending=False)
    .reset_index(drop=True)
)

top_results.head(10)

,mean_test_score,param_model__n_estimators,param_model__max_depth,param_model__min_samples_split,param_model__min_samples_leaf
0,0.834234,100,5,5,1
1,0.833948,200,5,2,1
2,0.833845,100,5,2,1
3,0.833161,100,10,5,1
4,0.832443,300,5,2,1
5,0.832047,200,5,5,1
6,0.831495,100,15,5,1
7,0.831492,100,None,5,1
8,0.829775,300,5,5,1
9,0.829733,100,15,2,1


In [45]:
# Hyperparameter Tuning Again using the previous result information for next one

params = {
    'model__n_estimators': [50,75,100,125,150,175,200],
    'model__max_depth': [1,2,4,5,6,8],
    'model__min_samples_split': [1,2,3,4,5],
    'model__min_samples_leaf': [1, 2,3,4]
}

grid = GridSearchCV(
    rf_pipe,
    params,
    cv=5,
    scoring='r2',
    n_jobs=-1
)

grid.fit(X_train, y_train)

print("Best Params:", grid.best_params_)
print("Best CV Score:", grid.best_score_)

C:\Users\abhinavkuriya\anaconda3\Lib\site-packages\sklearn\model_selection\_validation.py:528: FitFailedWarning: 
840 fits failed out of a total of 4200.
The score on these train-test partitions for these parameters will be set to nan.
If these failures are not expected, you can try to debug them by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
840 fits failed with the following error:
Traceback (most recent call last):
  File "C:\Users\abhinavkuriya\anaconda3\Lib\site-packages\sklearn\model_selection\_validation.py", line 866, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
    ~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\abhinavkuriya\anaconda3\Lib\site-packages\sklearn\base.py", line 1389, in wrapper
    return fit_method(estimator, *args, **kwargs)
  File "C:\Users\abhinavkuriya\anaconda3\Lib\site-packages\sklearn\pipeline.py", line 662, 

Best Params: {'model__max_depth': 6, 'model__min_samples_leaf': 1, 'model__min_samples_split': 3, 'model__n_estimators': 100}
Best CV Score: 0.8355018383617943


In [46]:
results = pd.DataFrame(grid.cv_results_)

top_results = (
    results[
        [
            'mean_test_score',
            'param_model__n_estimators',
            'param_model__max_depth',
            'param_model__min_samples_split',
            'param_model__min_samples_leaf'
        ]
    ]
    .sort_values('mean_test_score', ascending=False)
    .reset_index(drop=True)
)

top_results.head(10)

,mean_test_score,param_model__n_estimators,param_model__max_depth,param_model__min_samples_split,param_model__min_samples_leaf
0,0.835502,100,6,3,1
1,0.835077,150,5,2,1
2,0.835058,125,6,3,1
3,0.834863,150,6,3,1
4,0.834707,125,5,5,1
5,0.834387,125,5,2,1
6,0.834234,100,5,5,1
7,0.833973,150,5,5,1
8,0.833948,200,5,2,1
9,0.833857,125,5,3,1


In [49]:
print("Best Params:", grid.best_params_)

Best Params: {'model__max_depth': 6, 'model__min_samples_leaf': 1, 'model__min_samples_split': 3, 'model__n_estimators': 100}


In [51]:
# Best Model is 

categorical_cols = X.select_dtypes(include='object').columns


preprocessor = ColumnTransformer([
    ('cat',
     OneHotEncoder(drop='first', handle_unknown='ignore'),
     categorical_cols)
], remainder='passthrough')


rf_pipe = Pipeline([
    ('preprocess', preprocessor),
    ('model', RandomForestRegressor(
        n_estimators=100,
        max_depth=6,
        min_samples_leaf=1,
        min_samples_split=3,
        n_jobs=-1
    ))
])

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)


rf_pipe.fit(X_train, y_train)


print("Train R²:", rf_pipe.score(X_train, y_train))
print("Test R²:", rf_pipe.score(X_test, y_test))


cv_scores = cross_val_score(
    rf_pipe,
    X,
    y,
    cv=10,
    scoring='r2',
    n_jobs=-1
)

print("\nCV Scores:", cv_scores)
print("Mean CV R²:", cv_scores.mean())

Train R²: 0.9667072149652194
Test R²: 0.7973466506924081

CV Scores: [0.84766072 0.70214267 0.80731012 0.72353614 0.75112094 0.88071575
 0.81712811 0.91869152 0.93734492 0.7941172 ]
Mean CV R²: 0.817976808411936
